This notebook attempts to implement the order of purchases as a feature in our models. We will attempt similar models as prior notebooks, but using the order of purchases by each user and survey response IDs (which uniquely identify each user) as features.

In [1]:
%pip install pandas matplotlib seaborn scikit-learn tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 19.0 MB/s  0:00:18m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 27.2 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 44.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 34.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 44.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [tensorflow]7 [tensorflow]a]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf

I0000 00:00:1776657698.267841    9856 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776657699.248033    9856 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776657702.887127    9856 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


We first need to add the survey response IDs and order date columns back into our data. Because we did not change the order of our data or expand any rows, we can simply add these features from the unencoded data to the encoded data, so we do not need to rerun all of our encoding steps.

In [3]:
data_unencoded = pd.read_csv('/workspaces/group-project-bas-team/data/data_unencoded.csv')
data_encoded = pd.read_csv('/workspaces/group-project-bas-team/data/cleaned_data.csv')

In [4]:
data_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Columns: 156 entries, Purchase Price Per Unit to life-changes_Moved place of residence,Had a child
dtypes: float64(142), int64(14)
memory usage: 186.9 MB


In [5]:
data_unencoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Data columns (total 30 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Order Date                157026 non-null  str    
 1   Purchase Price Per Unit   157026 non-null  float64
 2   Quantity                  157026 non-null  int64  
 3   Shipping Address State    157026 non-null  str    
 4   Title                     157026 non-null  str    
 5   ASIN/ISBN (Product Code)  157026 non-null  str    
 6   Category                  157026 non-null  str    
 7   Survey ResponseID         157026 non-null  str    
 8   age                       157026 non-null  str    
 9   hispanic                  157026 non-null  str    
 10  race                      157026 non-null  str    
 11  education                 157026 non-null  str    
 12  income                    157026 non-null  str    
 13  gender                    157026 non-null  str    
 14 

In [6]:
data_encoded[['Survey ResponseID', 'Order Date']] = data_unencoded[['Survey ResponseID', 'Order Date']]

/tmp/ipykernel_9856/601039707.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_encoded[['Survey ResponseID', 'Order Date']] = data_unencoded[['Survey ResponseID', 'Order Date']]


In [7]:
data_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Columns: 158 entries, Purchase Price Per Unit to Order Date
dtypes: float64(142), int64(14), str(2)
memory usage: 189.3 MB


In [8]:
# storing data_encoded as just data for easier code writing
data = data_encoded

In [9]:
# Ensuring the data is sorted by order date within each user's order history
data['Order Date'] = pd.to_datetime(data['Order Date'])
data = data.sort_values(by=['Survey ResponseID', 'Order Date'])

Next, we need to create a column that assigns a number to each order, representing the position in the sequence of a customer's orders.

In [10]:
data['Purchase_Order'] = data.groupby('Survey ResponseID')['Order Date'].rank(method='first').astype(int)

/tmp/ipykernel_9856/693081789.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['Purchase_Order'] = data.groupby('Survey ResponseID')['Order Date'].rank(method='first').astype(int)


We will also add a column for time since last purchase

In [11]:
data['Days_Since_Last_Purchase'] = data.groupby('Survey ResponseID')['Order Date'].diff().dt.days.fillna(0)

/tmp/ipykernel_9856/646299313.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['Days_Since_Last_Purchase'] = data.groupby('Survey ResponseID')['Order Date'].diff().dt.days.fillna(0)


In [12]:
data.head()

,Purchase Price Per Unit,Quantity,Title,ASIN/ISBN (Product Code),Category,age,hispanic,education,income,howmany,...,"life-changes_Lost a job ,Moved place of residence,Became pregnant","life-changes_Lost a job ,Moved place of residence,Became pregnant,Had a child","life-changes_Lost a job ,Moved place of residence,Had a child",life-changes_Moved place of residence,"life-changes_Moved place of residence,Became pregnant,Had a child","life-changes_Moved place of residence,Had a child",Survey ResponseID,Order Date,Purchase_Order,Days_Since_Last_Purchase
0,7.98,1,83780,34099,586,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,R_01vNIayewjIIKMF,2018-12-04,1,0.0
1,13.99,1,16614,43470,729,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,R_01vNIayewjIIKMF,2018-12-22,2,18.0
2,10.45,1,73826,47343,432,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,R_01vNIayewjIIKMF,2018-12-25,3,3.0
3,10.00,1,77034,21023,1240,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,R_01vNIayewjIIKMF,2018-12-25,4,0.0
4,10.99,1,62681,39476,331,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,R_01vNIayewjIIKMF,2019-02-18,5,55.0


In [13]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Columns: 160 entries, Purchase Price Per Unit to Days_Since_Last_Purchase
dtypes: datetime64[us](1), float64(143), int64(15), str(1)
memory usage: 191.7 MB


Next, we will remove the Order Date column, since we only needed it to order purchases by date and we already have columns for day/month/year.

In [14]:
data = data.drop(columns=['Order Date'])

Finally, we will encode the survey response ID column. Since there are thousands of unique IDs in our dataset, we will simply label encode this column.

The rationale for keeping this column is to distinguish multiple purchases that may have the same rank in purchase order (i.e., the first purchases made by different users will both be ranked 1). This is feasible because our training and testing sets are split by date, so each unique order ID will be present in both the training and testing sets.

In [15]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
data['Survey ResponseID'] = encoder.fit_transform(data['Survey ResponseID'])

data.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Columns: 159 entries, Purchase Price Per Unit to Days_Since_Last_Purchase
dtypes: float64(143), int64(16)
memory usage: 190.5 MB


Now we can attempt to build models using these new columns. We will start by replicating our MLP.

In [16]:
from sklearn.preprocessing import StandardScaler

train = data[data['order_year']<=2021]
test = data[data['order_year']>2021]

data_limited_train = train.drop(['Title', 'ASIN/ISBN (Product Code)'], axis=1)
data_limited_test = test.drop(['Title', 'ASIN/ISBN (Product Code)'], axis=1)

X_train = data_limited_train.drop('Category', axis=1)
y_train = data_limited_train['Category']

X_test = data_limited_test.drop('Category', axis=1)
y_test = data_limited_test['Category']

X_train = X_train.drop(X_train.filter(regex='^Shipping Address').columns, axis=1)
X_test = X_test.drop(X_test.filter(regex='^Shipping Address').columns, axis=1)

X_train = X_train.drop(columns = ['diabetes_No', 'wheelchair_No', 'marijuana_No', 'sexual-orientation_heterosexual (straight)', 
                                  'gender_Female', 'alcohol_No'])
X_test = X_test.drop(columns = ['diabetes_No', 'wheelchair_No', 'marijuana_No', 'sexual-orientation_heterosexual (straight)', 
                                  'gender_Female', 'alcohol_No'])

X_train_scaled = StandardScaler().fit_transform(X_train)
X_test_scaled = StandardScaler().fit_transform(X_test)

In [17]:
X_train_scaled.shape[1]

101

In [18]:
y_train.max()

np.int64(1624)

In [19]:
import tensorflow as tf

tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(101, )))

model.add(tf.keras.layers.Dense(300, activation="relu")) 

model.add(tf.keras.layers.Dense(100, activation="relu"))

model.add(tf.keras.layers.Dense(1625, activation="softmax")) 

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
E0000 00:00:1776657714.111746    9856 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [20]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 300)            │        30,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1625)           │       164,125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 224,825 (878.22 KB)

 Trainable params: 224,825 (878.22 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])
              
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

In [22]:
history = model.fit(X_train_scaled, y_train, epochs=30, shuffle=False, validation_split=0.2, callbacks=callbacks)

Epoch 1/30


W0000 00:00:1776657714.895053    9856 cpu_allocator_impl.cc:82] Allocation of 36733296 exceeds 10% of free system memory.


2842/2842 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0589 - loss: 6.5902 - val_accuracy: 0.0557 - val_loss: 6.2758 - learning_rate: 0.0100
Epoch 2/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - accuracy: 0.0695 - loss: 6.1165 - val_accuracy: 0.0585 - val_loss: 6.2015 - learning_rate: 0.0100
Epoch 3/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.0744 - loss: 6.0373 - val_accuracy: 0.0553 - val_loss: 6.1787 - learning_rate: 0.0100
Epoch 4/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.0786 - loss: 5.9690 - val_accuracy: 0.0551 - val_loss: 6.1702 - learning_rate: 0.0100
Epoch 5/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 19s 4ms/step - accuracy: 0.0812 - loss: 5.8986 - val_accuracy: 0.0568 - val_loss: 6.1680 - learning_rate: 0.0100
Epoch 6/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0839 - loss: 5.8290 - val_accuracy: 0.0596 - val_loss: 6.1675 - learning_rate: 0.0100
Epoch 7/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.0864 - loss:

In [23]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1356/1356 - 4s - 3ms/step - accuracy: 0.0521 - loss: 6.1616

Test accuracy: 0.05206243693828583


In [24]:
# Same model, using Adam optimizer instead of SGD
tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(101, )))

model.add(tf.keras.layers.Dense(300, activation="relu")) 

model.add(tf.keras.layers.Dense(100, activation="relu"))

model.add(tf.keras.layers.Dense(1625, activation="softmax")) 

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

In [25]:
history = model.fit(X_train_scaled, y_train, epochs=30, shuffle=False, validation_split=0.2, callbacks=callbacks)

Epoch 1/30


W0000 00:00:1776657893.457972    9856 cpu_allocator_impl.cc:82] Allocation of 36733296 exceeds 10% of free system memory.


2842/2842 ━━━━━━━━━━━━━━━━━━━━ 19s 6ms/step - accuracy: 0.0574 - loss: 6.2253 - val_accuracy: 0.0611 - val_loss: 6.2955 - learning_rate: 0.0010
Epoch 2/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 19s 7ms/step - accuracy: 0.0708 - loss: 5.8648 - val_accuracy: 0.0615 - val_loss: 6.3601 - learning_rate: 0.0010
Epoch 3/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - accuracy: 0.0789 - loss: 5.6217 - val_accuracy: 0.0563 - val_loss: 6.3818 - learning_rate: 0.0010
Epoch 4/30
2828/2842 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0691 - loss: 5.4526
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - accuracy: 0.0833 - loss: 5.4352 - val_accuracy: 0.0501 - val_loss: 6.5960 - learning_rate: 0.0010
Epoch 5/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - accuracy: 0.0873 - loss: 5.3201 - val_accuracy: 0.0465 - val_loss: 6.4906 - learning_rate: 5.0000e-04
Epoch 6/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.0944 - los

In [26]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1356/1356 - 5s - 4ms/step - accuracy: 0.0406 - loss: 6.3897

Test accuracy: 0.040649279952049255


In [27]:
X_train_scaled.shape

(113655, 101)

In [28]:
X_train_reshaped = X_train_scaled.reshape((X_train_scaled.shape[0], 1, 101))
X_test_reshaped = X_test_scaled.reshape((X_test_scaled.shape[0], 1, 101))

In [29]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(1, 101))) 

model.add(tf.keras.layers.LSTM(150, return_sequences=False)) 

model.add(tf.keras.layers.Dense(1625, activation="softmax"))

In [30]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 150)            │       151,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1625)           │       245,375 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 396,575 (1.51 MB)

 Trainable params: 396,575 (1.51 MB)

 Non-trainable params: 0 (0.00 B)

In [31]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

In [32]:
history = model.fit(X_train_reshaped, y_train, epochs=30, shuffle=False, validation_split=0.2, callbacks=callbacks)

Epoch 1/30


W0000 00:00:1776658005.599021    9856 cpu_allocator_impl.cc:82] Allocation of 36733296 exceeds 10% of free system memory.


2842/2842 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - accuracy: 0.0497 - loss: 7.3107 - val_accuracy: 0.0612 - val_loss: 7.2222 - learning_rate: 0.0100
Epoch 2/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - accuracy: 0.0540 - loss: 7.1464 - val_accuracy: 0.0612 - val_loss: 7.0649 - learning_rate: 0.0100
Epoch 3/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.0540 - loss: 7.0025 - val_accuracy: 0.0612 - val_loss: 6.9472 - learning_rate: 0.0100
Epoch 4/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - accuracy: 0.0540 - loss: 6.8950 - val_accuracy: 0.0612 - val_loss: 6.8688 - learning_rate: 0.0100
Epoch 5/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - accuracy: 0.0540 - loss: 6.8044 - val_accuracy: 0.0612 - val_loss: 6.8060 - learning_rate: 0.0100
Epoch 6/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - accuracy: 0.0562 - loss: 6.7147 - val_accuracy: 0.0608 - val_loss: 6.7516 - learning_rate: 0.0100
Epoch 7/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - accuracy: 0.0609 - loss:

In [33]:
test_loss, test_acc = model.evaluate(X_test_reshaped, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1356/1356 - 5s - 3ms/step - accuracy: 0.0549 - loss: 6.1844

Test accuracy: 0.05492149293422699


In [34]:
# Adding more layers

tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(1, 101))) 

model.add(tf.keras.layers.LSTM(150, return_sequences=True)) 
model.add(tf.keras.layers.LSTM(100, return_sequences=True)) 
model.add(tf.keras.layers.LSTM(100, return_sequences=False)) 

model.add(tf.keras.layers.Dense(1625, activation="softmax"))

In [35]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 1, 150)         │       151,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 1, 100)         │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 100)            │        80,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1625)           │       164,125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 496,125 (1.89 MB)

 Trainable params: 496,125 (1.89 MB)

 Non-trainable params: 0 (0.00 B)

In [36]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

In [37]:
history = model.fit(X_train_reshaped, y_train, epochs=30, shuffle=False, validation_split=0.2, callbacks=callbacks)

Epoch 1/30


W0000 00:00:1776658518.789844    9856 cpu_allocator_impl.cc:82] Allocation of 36733296 exceeds 10% of free system memory.


2842/2842 ━━━━━━━━━━━━━━━━━━━━ 24s 8ms/step - accuracy: 0.0538 - loss: 7.3133 - val_accuracy: 0.0612 - val_loss: 7.2167 - learning_rate: 0.0100
Epoch 2/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 22s 8ms/step - accuracy: 0.0540 - loss: 7.1604 - val_accuracy: 0.0612 - val_loss: 7.0562 - learning_rate: 0.0100
Epoch 3/30
2838/2842 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0457 - loss: 7.0869
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.0540 - loss: 7.0317 - val_accuracy: 0.0612 - val_loss: 6.9319 - learning_rate: 0.0100
Epoch 4/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - accuracy: 0.0540 - loss: 6.9586 - val_accuracy: 0.0612 - val_loss: 6.8852 - learning_rate: 0.0050
Epoch 5/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - accuracy: 0.0540 - loss: 6.9197 - val_accuracy: 0.0612 - val_loss: 6.8440 - learning_rate: 0.0050


In [38]:
test_loss, test_acc = model.evaluate(X_test_reshaped, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1356/1356 - 5s - 4ms/step - accuracy: 0.0395 - loss: 7.2576

Test accuracy: 0.0395425520837307
